<a href="https://colab.research.google.com/github/MalakSalehh/Thesis-datasets/blob/main/FINAL_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q openai pandas numpy scipy scikit-learn

In [ ]:
import re
import json
import time
import numpy as np
import pandas as pd
from openai import OpenAI

In [ ]:
GROQ_API_KEY = "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"

client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

In [ ]:
url = "https://raw.githubusercontent.com/MalakSalehh/Thesis-datasets/main/training_set_rel3.tsv"
df = pd.read_csv(url, sep="\t", encoding="latin1")
print(df.shape)
df.head()

(12976, 28)


,essay_id,essay_set,essay,rater1_domain1,rater2_domain1,rater3_domain1,domain1_score,rater1_domain2,rater2_domain2,domain2_score,...,rater2_trait3,rater2_trait4,rater2_trait5,rater2_trait6,rater3_trait1,rater3_trait2,rater3_trait3,rater3_trait4,rater3_trait5,rater3_trait6
0,1,1,"Dear local newspaper, I think effects computer...",4,4,NaN,8,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,1,"Dear @CAPS1 @CAPS2, I believe that using compu...",5,4,NaN,9,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,1,"Dear, @CAPS1 @CAPS2 @CAPS3 More and more peopl...",4,3,NaN,7,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,1,"Dear Local Newspaper, @CAPS1 I have found that...",5,5,NaN,10,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,1,"Dear @LOCATION1, I know having computers has a...",4,4,NaN,8,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
set1 = df[df["essay_set"] == 1].copy()
set1 = set1[["essay_id", "essay", "domain1_score"]]

set1["essay"] = set1["essay"].str.replace("\n", " ", regex=False)
set1["essay"] = set1["essay"].str.strip()

print(set1.shape)
set1.head()

(1783, 3)


,essay_id,essay,domain1_score
0,1,"Dear local newspaper, I think effects computer...",8
1,2,"Dear @CAPS1 @CAPS2, I believe that using compu...",9
2,3,"Dear, @CAPS1 @CAPS2 @CAPS3 More and more peopl...",7
3,4,"Dear Local Newspaper, @CAPS1 I have found that...",10
4,5,"Dear @LOCATION1, I know having computers has a...",8


In [ ]:
def categorize(score):
    if score <= 5:
        return "Low"
    elif score <= 8:
        return "Medium"
    else:
        return "High"

set1["score_category"] = set1["domain1_score"].apply(categorize)
set1["domain1_score"].value_counts().sort_index()

,count
domain1_score,
2,10
3,1
4,17
5,17
6,110
7,135
8,687
9,334
10,316


In [ ]:
cal_low = set1[set1["score_category"] == "Low"].sample(2, random_state=42)
cal_med = set1[set1["score_category"] == "Medium"].sample(2, random_state=42)
cal_high = set1[set1["score_category"] == "High"].sample(2, random_state=42)

calibration = pd.concat([cal_low, cal_med, cal_high]).sample(frac=1, random_state=42).reset_index(drop=True)
calibration

,essay_id,essay,domain1_score,score_category
0,1592,The computers are cool. Do you now I werpsite ...,2,Low
1,995,"I think computers are good, because some peopl...",4,Low
2,686,"Dear local newspaper, I would like to speak up...",9,High
3,1572,"Dear to @CAPS1 this @MONTH1 concern, Computers...",8,Medium
4,1567,"Dear @ORGANIZATION1, @DATE1, everywhere you lo...",11,High
5,146,Dear local newspaper I think that usieng compu...,6,Medium


In [ ]:
calibration_ids = calibration["essay_id"].tolist()
demo = set1[~set1["essay_id"].isin(calibration_ids)].sample(30, random_state=42).reset_index(drop=True)
demo

,essay_id,essay,domain1_score,score_category
0,66,Wouldnt it be great to use computers in everyd...,9,High
1,946,According to the @ORGANIZATION1 govermant @PER...,8,Medium
2,837,"Dear, @ORGANIZATION1 the big rectangular shape...",8,Medium
3,1710,Wouldn't you agree that computers have a great...,9,High
4,804,"Dear Local Newspaper, The effects computers ha...",8,Medium
5,1539,"Dear Local Newspaper, @CAPS1 personal experien...",9,High
6,1042,"Dear @ORGANIZATION1, I took a poll of @NUM1 pe...",8,Medium
7,241,Dear local @ORGANIZATION1 @CAPS1 of @LOCATION1...,9,High
8,1211,Dear editor: @CAPS1 you know that @PERCENT1 of...,9,High
9,1485,"Dear local newspaper, did you know that just a...",9,High


In [ ]:
def format_calibration_examples(calibration_df):
    examples = ""
    for i, row in calibration_df.iterrows():
        examples += f"""
Calibration Example {i+1}

Essay:
{row['essay']}

Human Score: {row['domain1_score']}
Category: {row['score_category']}
"""
    return examples

calibration_text = format_calibration_examples(calibration)

In [ ]:
rubric_text = """
Essay Prompt:
Write a letter to your local newspaper stating your opinion on the effects computers have on people and persuade readers to agree with your position.

Rubric:
1 – Undeveloped response with minimal support.
2 – Underdeveloped response with weak reasoning.
3 – Minimally developed argument with limited support.
4 – Adequate argument with some supporting details.
5 – Developed argument with clear reasoning and organization.
6 – Strong persuasive response with well-developed ideas.

Scoring rule:
Two raters assign scores from 1–6.
Final score = sum of both scores.
Final score range: 2–12.
"""

In [ ]:
def build_baseline_prompt(essay_text):
    return f"""
You are an expert essay grader evaluating student essays.

{rubric_text}

Calibration Examples:
{calibration_text}

Now evaluate the following essay.

Essay:
{essay_text}

Instructions:
1. Predict the final score between 2 and 12.
2. Predict the category: Low, Medium, or High.
3. Explain your reasoning.

Important:
Base your scoring strictly on the rubric and calibration examples.

Return your answer in this format:

Final Score:
Category:
Reasoning:
"""

In [ ]:
def get_response(prompt):

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",  # Working as of April 2026
        messages=[
            {"role": "system", "content": "You are an expert essay grader. Follow the rubric strictly."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3,  # Lower temperature for more consistent scoring
        max_tokens=4096
    )
    return response.choices[0].message.content

# Re-run your 5 essays
results = []

for i, row in demo.iterrows():
    essay_id = row["essay_id"]
    essay_text = row["essay"]
    human_score = row["domain1_score"]

    prompt = build_baseline_prompt(essay_text)
    output = get_response(prompt)

    results.append({
        "essay_id": essay_id,
        "human_score": human_score,
        "model_output": output
    })

    print(f"Done Essay {essay_id}")
    time.sleep(1)

Done Essay 66
Done Essay 946
Done Essay 837
Done Essay 1710
Done Essay 804
Done Essay 1539
Done Essay 1042
Done Essay 241
Done Essay 1211
Done Essay 1485
Done Essay 941
Done Essay 1126
Done Essay 1564
Done Essay 1730
Done Essay 1073
Done Essay 922
Done Essay 110
Done Essay 1588
Done Essay 558
Done Essay 886
Done Essay 355
Done Essay 402
Done Essay 1519
Done Essay 1402
Done Essay 325
Done Essay 536
Done Essay 1351
Done Essay 699
Done Essay 1691
Done Essay 738


In [ ]:
results_df = pd.DataFrame(results)
results_df

,essay_id,human_score,model_output
0,66,9,**Final Score:** 9 \n**Category:** High \n\n...
1,946,8,**Final Score:** 6 \n**Category:** Medium \n...
2,837,8,**Final Score:** 8 \n**Category:** Medium \n...
3,1710,9,**Final Score:** 8 \n**Category:** Medium \n...
4,804,8,**Final Score:** 9 \n**Category:** Medium \n...
5,1539,9,**Final Score:** 8 \n**Category:** Medium \n...
6,1042,8,**Final Score:** 8 \n**Category:** Medium \n...
7,241,9,**Final Score:** 6 \n**Category:** Medium \n...
8,1211,9,**Final Score:** 10 \n**Category:** High \n\...
9,1485,9,**Final Score:** 8 \n**Category:** Medium \n...


In [ ]:
import re

def extract_score(text):
    if pd.isna(text):
        return None

    match = re.search(r"\*{0,2}Final Score:\*{0,2}\s*(\d+)", str(text), re.IGNORECASE)
    if match:
        return int(match.group(1))
    return None

def extract_category(text):
    if pd.isna(text):
        return None

    match = re.search(r"\*{0,2}Category:\*{0,2}\s*(Low|Medium|High)", str(text), re.IGNORECASE)
    if match:
        return match.group(1).capitalize()
    return None

results_df["predicted_score"] = results_df["model_output"].apply(extract_score)
results_df["predicted_category"] = results_df["model_output"].apply(extract_category)

results_df[["essay_id", "human_score", "model_output", "predicted_score", "predicted_category"]]

,essay_id,human_score,model_output,predicted_score,predicted_category
0,66,9,**Final Score:** 9 \n**Category:** High \n\n...,9,High
1,946,8,**Final Score:** 6 \n**Category:** Medium \n...,6,Medium
2,837,8,**Final Score:** 8 \n**Category:** Medium \n...,8,Medium
3,1710,9,**Final Score:** 8 \n**Category:** Medium \n...,8,Medium
4,804,8,**Final Score:** 9 \n**Category:** Medium \n...,9,Medium
5,1539,9,**Final Score:** 8 \n**Category:** Medium \n...,8,Medium
6,1042,8,**Final Score:** 8 \n**Category:** Medium \n...,8,Medium
7,241,9,**Final Score:** 6 \n**Category:** Medium \n...,6,Medium
8,1211,9,**Final Score:** 10 \n**Category:** High \n\...,10,High
9,1485,9,**Final Score:** 8 \n**Category:** Medium \n...,8,Medium


In [ ]:
clean_df = results_df.dropna(subset=["predicted_score"]).copy()

from sklearn.metrics import cohen_kappa_score, mean_absolute_error
from scipy.stats import pearsonr

qwk = cohen_kappa_score(clean_df["human_score"], clean_df["predicted_score"], weights="quadratic")
pcc, _ = pearsonr(clean_df["human_score"], clean_df["predicted_score"])
mae = mean_absolute_error(clean_df["human_score"], clean_df["predicted_score"])

print("QWK:", qwk)
print("PCC:", pcc)
print("MAE:", mae)

QWK: 0.41978021978021984
PCC: 0.4804521940483344
MAE: 1.3666666666666667


In [ ]:
from sklearn.metrics import cohen_kappa_score, mean_absolute_error
from scipy.stats import pearsonr

qwk = cohen_kappa_score(
    clean_df["human_score"],
    clean_df["predicted_score"],
    weights="quadratic"
)

pcc, _ = pearsonr(
    clean_df["human_score"],
    clean_df["predicted_score"]
)

mae = mean_absolute_error(
    clean_df["human_score"],
    clean_df["predicted_score"]
)

print("QWK:", qwk)
print("PCC:", pcc)
print("MAE:", mae)

QWK: 0.41978021978021984
PCC: 0.4804521940483344
MAE: 1.3666666666666667


# **IGNORE THE ABOVE , WE ARE TRYING FINE TUNING**

In [2]:
import pandas as pd

# Load dataset
url = "https://raw.githubusercontent.com/MalakSalehh/Thesis-datasets/main/training_set_rel3.tsv"
df = pd.read_csv(url, sep="\t", encoding="latin1")
print(df.shape)
df.head()

# Keep only essay set 1
set1_df = df[df["essay_set"] == 1].copy()

# Keep only important columns
set1_df = set1_df[["essay_id", "essay_set", "essay", "domain1_score"]].copy()

# Clean essay text
set1_df["essay"] = (
    set1_df["essay"]
    .astype(str)
    .str.replace("\n", " ", regex=False)
    .str.strip()
)

# Remove empty essays if any
set1_df = set1_df[set1_df["essay"].str.len() > 0].copy()

print(set1_df.shape)
set1_df.head()

(12976, 28)
(1783, 4)


,essay_id,essay_set,essay,domain1_score
0,1,1,"Dear local newspaper, I think effects computer...",8
1,2,1,"Dear @CAPS1 @CAPS2, I believe that using compu...",9
2,3,1,"Dear, @CAPS1 @CAPS2 @CAPS3 More and more peopl...",7
3,4,1,"Dear Local Newspaper, @CAPS1 I have found that...",10
4,5,1,"Dear @LOCATION1, I know having computers has a...",8


In [3]:
def score_to_category(score):
    if 2 <= score <= 5:
        return "Low"
    elif 6 <= score <= 8:
        return "Medium"
    elif 9 <= score <= 12:
        return "High"
    else:
        return "Unknown"

set1_df["category"] = set1_df["domain1_score"].apply(score_to_category)

set1_df[["essay_id", "domain1_score", "category"]].head()

,essay_id,domain1_score,category
0,1,8,Medium
1,2,9,High
2,3,7,Medium
3,4,10,High
4,5,8,Medium


In [4]:
essay_prompt_text = """
Write a letter to your local newspaper stating your opinion on the effects computers have on people and persuade readers to agree with your position.
""".strip()

rubric_text = """
The essay should be evaluated based on persuasive writing quality.
Score the response according to how clearly it presents a position, how well ideas are developed, how effectively details support the argument, and how organized and coherent the essay is.

Scoring interpretation:
1 = very weak response with minimal development
2 = limited response with weak support
3 = basic response with some development
4 = adequate response with reasonable organization and support
5 = strong response with clear development and effective support
6 = very strong persuasive response with clear organization, strong development, and effective language use

Final score rule:
Two human raters each assign a score from 1 to 6.
The final score is the sum of both ratings, so the final score ranges from 2 to 12.

Category mapping:
2-5 = Low
6-8 = Medium
9-12 = High
""".strip()

print(essay_prompt_text)
print()
print(rubric_text)

Write a letter to your local newspaper stating your opinion on the effects computers have on people and persuade readers to agree with your position.

The essay should be evaluated based on persuasive writing quality.
Score the response according to how clearly it presents a position, how well ideas are developed, how effectively details support the argument, and how organized and coherent the essay is.

Scoring interpretation:
1 = very weak response with minimal development
2 = limited response with weak support
3 = basic response with some development
4 = adequate response with reasonable organization and support
5 = strong response with clear development and effective support
6 = very strong persuasive response with clear organization, strong development, and effective language use

Final score rule:
Two human raters each assign a score from 1 to 6.
The final score is the sum of both ratings, so the final score ranges from 2 to 12.

Category mapping:
2-5 = Low
6-8 = Medium
9-12 = Hi

In [5]:
def build_input_text(essay_text):
    return f"""You are an expert essay evaluator.

Evaluate the following student essay using the provided essay prompt and rubric.

Essay Prompt:
{essay_prompt_text}

Rubric:
{rubric_text}

Student Essay:
{essay_text}
""".strip()

def build_output_text(score, category):
    return f"""Score: {score}
Category: {category}
Reasoning: This essay was assigned a {category.lower()} score based on its overall alignment with the rubric criteria for content, support, and organization."""

In [6]:
set1_df["input_text"] = set1_df["essay"].apply(build_input_text)
set1_df["output_text"] = set1_df.apply(
    lambda row: build_output_text(row["domain1_score"], row["category"]),
    axis=1
)

set1_df[["essay_id", "input_text", "output_text"]].head(1)

,essay_id,input_text,output_text
0,1,You are an expert essay evaluator.\n\nEvaluate...,Score: 8\nCategory: Medium\nReasoning: This es...


In [7]:
##split the data into train and test
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    set1_df[["essay_id", "input_text", "output_text", "domain1_score", "category"]],
    test_size=0.2,
    random_state=42,
    stratify=set1_df["category"]
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.head()

Train shape: (1426, 5)
Test shape: (357, 5)


,essay_id,input_text,output_text,domain1_score,category
0,1690,You are an expert essay evaluator.\n\nEvaluate...,Score: 9\nCategory: High\nReasoning: This essa...,9,High
1,69,You are an expert essay evaluator.\n\nEvaluate...,Score: 7\nCategory: Medium\nReasoning: This es...,7,Medium
2,943,You are an expert essay evaluator.\n\nEvaluate...,Score: 9\nCategory: High\nReasoning: This essa...,9,High
3,905,You are an expert essay evaluator.\n\nEvaluate...,Score: 6\nCategory: Medium\nReasoning: This es...,6,Medium
4,188,You are an expert essay evaluator.\n\nEvaluate...,Score: 9\nCategory: High\nReasoning: This essa...,9,High


  ## - **convert to Hugging Face dataset format**



In [1]:
!pip install -q datasets
!pip uninstall -y pyarrow datasets
!pip install pyarrow==15.0.2 datasets==2.19.1

Found existing installation: pyarrow 15.0.2
Uninstalling pyarrow-15.0.2:
  Successfully uninstalled pyarrow-15.0.2
Found existing installation: datasets 2.19.1
Uninstalling datasets-2.19.1:
  Successfully uninstalled datasets-2.19.1
  Using cached pyarrow-15.0.2-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
  Using cached datasets-2.19.1-py3-none-any.whl.metadata (19 kB)
Using cached pyarrow-15.0.2-cp312-cp312-manylinux_2_28_x86_64.whl (38.3 MB)
Using cached datasets-2.19.1-py3-none-any.whl (542 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
trl 1.0.0 requires datasets>=4.7.0, but you have datasets 2.19.1 which is incompatible.


In [8]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df[["input_text", "output_text"]])
test_dataset = Dataset.from_pandas(test_df[["input_text", "output_text"]])

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['input_text', 'output_text'],
    num_rows: 1426
})
Dataset({
    features: ['input_text', 'output_text'],
    num_rows: 357
})


- **Change the columns into instruction-tuning format**

      - Most fine-tuning setups want one combined text field

In [9]:
def format_example(example):
    return {
        "text": f"""### Instruction:
{example['input_text']}

### Response:
{example['output_text']}"""
    }

train_dataset = train_dataset.map(format_example)
test_dataset = test_dataset.map(format_example)

print(train_dataset[0]["text"])

Map:   0%|          | 0/1426 [00:00<?, ? examples/s]

Map:   0%|          | 0/357 [00:00<?, ? examples/s]

### Instruction:
You are an expert essay evaluator.

Evaluate the following student essay using the provided essay prompt and rubric.

Essay Prompt:
Write a letter to your local newspaper stating your opinion on the effects computers have on people and persuade readers to agree with your position.

Rubric:
The essay should be evaluated based on persuasive writing quality.
Score the response according to how clearly it presents a position, how well ideas are developed, how effectively details support the argument, and how organized and coherent the essay is.

Scoring interpretation:
1 = very weak response with minimal development
2 = limited response with weak support
3 = basic response with some development
4 = adequate response with reasonable organization and support
5 = strong response with clear development and effective support
6 = very strong persuasive response with clear organization, strong development, and effective language use

Final score rule:
Two human raters each assign

In [2]:
## install the training libraries
!pip install -q transformers peft accelerate bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 19.7 MB/s eta 0:00:00


In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "microsoft/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [11]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["qkv_proj", "o_proj", "gate_up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 12,582,912 || all params: 3,833,662,464 || trainable%: 0.3282


In [12]:
def tokenize_function(example):
    result = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    result["labels"] = result["input_ids"].copy()
    return result

tokenized_train = train_dataset.map(tokenize_function, batched=False)
tokenized_test = test_dataset.map(tokenize_function, batched=False)

Map:   0%|          | 0/1426 [00:00<?, ? examples/s]

Map:   0%|          | 0/357 [00:00<?, ? examples/s]

In [13]:
tokenized_train = tokenized_train.remove_columns(
    [col for col in tokenized_train.column_names if col not in ["input_ids", "attention_mask", "labels"]]
)

tokenized_test = tokenized_test.remove_columns(
    [col for col in tokenized_test.column_names if col not in ["input_ids", "attention_mask", "labels"]]
)

print(tokenized_train)
print(tokenized_test)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1426
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 357
})


In [14]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="/content/fine_tuned_phi3_aes",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="no",
    fp16=True,
    report_to="none"
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator
)

In [18]:
trainer.train()

Step,Training Loss
10,1.898841
20,1.268865
30,1.101760
40,1.162764
50,1.118852
60,1.110891
70,1.116998
80,1.040881
90,1.093281
100,1.085991


TrainOutput(global_step=357, training_loss=1.0909316225880001, metrics={'train_runtime': 1143.5778, 'train_samples_per_second': 1.247, 'train_steps_per_second': 0.312, 'total_flos': 1.6362518958047232e+16, 'train_loss': 1.0909316225880001, 'epoch': 1.0})

In [19]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [20]:
save_path = "/content/drive/MyDrive/phi3_aes_final"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

('/content/drive/MyDrive/phi3_aes_final/tokenizer_config.json',
 '/content/drive/MyDrive/phi3_aes_final/chat_template.jinja',
 '/content/drive/MyDrive/phi3_aes_final/tokenizer.json')

In [ ]:
model.save_pretrained("/content/fine_tuned_phi3_aes_final")
tokenizer.save_pretrained("/content/fine_tuned_phi3_aes_final")

('/content/fine_tuned_phi3_aes_final/tokenizer_config.json',
 '/content/fine_tuned_phi3_aes_final/chat_template.jinja',
 '/content/fine_tuned_phi3_aes_final/tokenizer.json')

In [22]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

In [23]:
sample_prompt = f"""### Instruction:
{test_df.loc[0, "input_text"]}

### Response:
"""

In [ ]:
result = generator(
    sample_prompt,
    max_new_tokens=120,
    do_sample=False,
    return_full_text=False
)[0]["generated_text"]

print(result)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
sample_prompt = f"""### Instruction:
{test_df.loc[0, "input_text"]}

### Response:
"""

inputs = tokenizer(sample_prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False,
        repetition_penalty=1.2,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

result = tokenizer.decode(outputs[0], skip_special_tokens=True)
response_only = result.split("### Response:")[-1].strip()

print(response_only)

In [ ]:

import re

def extract_score(text):
    match = re.search(r"Score:\s*(\d+)", text)
    return int(match.group(1)) if match else None

def extract_category(text):
    match = re.search(r"Category:\s*(Low|Medium|High)", text, re.IGNORECASE)
    return match.group(1).capitalize() if match else None

print("Predicted score:", extract_score(response_only))
print("Predicted category:", extract_category(response_only))

Predicted score: None
Predicted category: None


In [ ]:
import json

path = "/content/FINAL.ipynb"   # change if needed

with open(path, "r", encoding="utf-8") as f:
    nb = json.load(f)

# remove broken widgets
nb.get("metadata", {}).pop("widgets", None)

for cell in nb.get("cells", []):
    if "metadata" in cell and "widgets" in cell["metadata"]:
        del cell["metadata"]["widgets"]

with open(path, "w", encoding="utf-8") as f:
    json.dump(nb, f, ensure_ascii=False, indent=1)

print("✅ FIXED")

FileNotFoundError: [Errno 2] No such file or directory: '/content/FINAL.ipynb'

In [ ]:
import os

for root, dirs, files in os.walk("/content"):
    for file in files:
        if file.endswith(".ipynb"):
            print(os.path.join(root, file))